In [2]:
from pathlib import Path
import sys

parent_root=Path.cwd().parents[1]
sys.path.append(str(parent_root))

In [3]:
from src.query_rewriter.models import (QueryRewriteRequest, ExpansionResult)
from src.query_rewriter.base import BaseQueryRewriter
from src.query_rewriter.models import ChatMessage
from langchain_core.messages import(HumanMessage, BaseMessage, SystemMessage, AIMessage)
from src.query_rewriter.prompts import EXPANSION_SYSTEM_PROMPT

import re
from src.query_rewriter.utils import (llm_util, message_util)


In [ ]:
class Expansion(BaseQueryRewriter):

    def __init__(self, llm):
        self.llm=llm

    def _build_messages(self, request: QueryRewriteRequest)-> list[BaseMessage]:
        messages: list[BaseMessage]= [SystemMessage(content= EXPANSION_SYSTEM_PROMPT )]

        messages.extend(message_util.build_messages(history=request.history))

        
        messages.append(
            HumanMessage(
    content=f"""
    Current Query:

    {request.query}

    Generate multiple expanded search queries.
    Return one query per line.
    """
    )
        )

        return messages


    def _parse(self, response) -> list[str]:
        return [
            re.sub(r"^(\d+\.\s*|[-*]\s*)", "", line).strip()
            for line in response.content.splitlines()
            if line.strip()
        ]
    
    def _validate(self, expanded_queries:list):
        if not expanded_queries:
            raise ValueError(" there were no expanded queries it is empty")
        return expanded_queries

    def rewrite(self, request: QueryRewriteRequest):

        messages=self._build_messages(request=request)
        response=llm_util.invoke_llm( llm=self.llm, messages=messages)
        expanded_queries=self._parse(response)
        expanded_queries=self._validate(expanded_queries)

        return ExpansionResult(original_query=request.query, expanded_queries=expanded_queries )
        
